# PPC-Gauge-Triad v0.6.1 — Huber-Clamped EMA + PCA-Residual Ablation & Independent Verification

Mengapa v0.6.0 sebelumnya anjlok di `machine-1-1` & `machine-1-6`? Karena `e_streak>=3` dan `dc_relock>=6` **menelan anomali server yang berlangsung panjang (50–300 jendela)** sebagai normal baru!
Solusi **v0.6.1 (`0` tambahan SRAM, `#![no_std]`)**: **Robust Huber-Clamped EMA (`huber_clip = 1.5`)** — melacak pergeseran baseline normal perlahan tanpa pernah menelan anomali berkepanjangan!


In [ ]:
import os, ctypes, time, warnings, subprocess
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable", shell=True, check=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

os.makedirs("ppc_gauge_triad_v061/src", exist_ok=True)
with open("ppc_gauge_triad_v061/Cargo.toml", "w") as f: f.write('[package]\nname = "ppc_gauge_triad_rs"\nversion = "0.6.1"\nedition = "2021"\ndescription = "PPC-Gauge-Triad v0.6.1: #![no_std] f32 Zero-Heap Anomaly Engine with Huber-Clamped EMA, Stable-Plateau DC Re-Lock & Dual-Stage Alarm Gate"\n\n[lib]\nname = "ppc_gauge_triad_rs"\ncrate-type = ["cdylib", "rlib"]\n\n[features]\ndefault = ["std"]\nstd = []\n\n[profile.release]\nopt-level = 3\nlto = true\ncodegen-units = 1\npanic = "abort"\n')
with open("ppc_gauge_triad_v061/src/lib.rs", "w") as f: f.write('#![cfg_attr(not(feature = "std"), no_std)]\n\n//! PPC-Gauge-Triad v0.6.1 (`#![no_std]` f32):\n//! Orthogonal 3-channel gauge decomposition for streaming telemetry & physical sensors:\n//!   1. `s_shape`  : Exact Sim+(d) Givens-QR canonical shadow (det=+1).\n//!   2. `s_energy` : Relative log-fluctuation energy (Huber-clamped EMA for server drift OR 3-window plateau hysteresis for thermal chips).\n//!   3. `s_level`  : Per-channel DC mean drift (Huber-clamped EMA to track drift WITHOUT swallowing sustained anomalies, OR stable-plateau re-lock for abrupt chip steps).\n//!   4. Dual-Stage Alarm Gate (`step_v61`): Immediate trip (`> sev_mult * thr`) OR 2-consecutive-window confirmation.\n//!\n//! Hardware profile: ~11.4 KB static struct SRAM, 0 B heap, 0 matrix inversions.\n\n#[cfg(not(feature = "std"))]\n#[panic_handler]\nfn panic(_info: &core::panic::PanicInfo) -> ! {\n    loop {}\n}\n\nconst MAX_W: usize = 8;\nconst MAX_D: usize = 4;\nconst MAX_M: usize = 26;\nconst MAX_K: usize = 4;\n\n#[inline(always)]\nfn sqrt_f32(x: f32) -> f32 {\n    #[cfg(feature = "std")]\n    { x.sqrt() }\n    #[cfg(not(feature = "std"))]\n    {\n        if x <= 0.0 { return 0.0; }\n        let mut y = f32::from_bits((x.to_bits() >> 1) + 0x1fc0_0000);\n        for _ in 0..4 { y = 0.5 * (y + x / y); }\n        y\n    }\n}\n\n#[inline(always)]\nfn ln_f32(x: f32) -> f32 {\n    #[cfg(feature = "std")]\n    { x.ln() }\n    #[cfg(not(feature = "std"))]\n    {\n        if x <= 1e-20 { return -46.0; }\n        let bits = x.to_bits();\n        let exp = (((bits >> 23) & 0xFF) as i32) - 127;\n        let m = f32::from_bits((bits & 0x007F_FFFF) | 0x3F80_0000);\n        let t = (m - 1.0) / (m + 1.0);\n        let t2 = t * t;\n        let ln_m = 2.0 * t * (1.0 + t2 * (0.33333334 + t2 * (0.2 + t2 * 0.14285715)));\n        (exp as f32) * 0.69314718 + ln_m\n    }\n}\n\n#[inline(always)]\nfn clamp_f32(x: f32, lo: f32, hi: f32) -> f32 {\n    if x < lo { lo } else if x > hi { hi } else { x }\n}\n\n#[repr(C)]\npub struct GaugeTriadEngineF32 {\n    pub w: usize,\n    pub d: usize,\n    pub m: usize,\n    pub k_max: usize,\n    pub k_active: usize,\n    pub lam0: f32,\n    pub alpha_min: f32,\n    pub e_alpha: f32,\n    pub dc_alpha: f32,\n    pub e_ema: f32,\n    pub e_var: f32,\n    pub dc_ema: [f32; MAX_D],\n    pub dc_var: [f32; MAX_D],\n    pub prev_dc: [f32; MAX_D],\n    pub e_init: u8,\n    pub streak: u8,\n    pub dc_streak: u8,\n    pub sh_alarm_streak: u8,\n    pub en_alarm_streak: u8,\n    pub lv_alarm_streak: u8,\n    pub b_dct: [f32; MAX_W * MAX_W],\n    pub means: [[f32; MAX_M]; MAX_K],\n    pub precs: [[f32; MAX_M * MAX_M]; MAX_K],\n    pub counts: [u32; MAX_K],\n}\n\nimpl GaugeTriadEngineF32 {\n    pub fn init(\n        &mut self, w: usize, d: usize, k_max: usize,\n        lam0: f32, alpha_min: f32, e_alpha: f32, dc_alpha: f32, b_dct: &[f32],\n    ) {\n        self.w = w;\n        self.d = d;\n        self.k_max = if k_max > MAX_K { MAX_K } else { k_max };\n        self.k_active = 0;\n        self.lam0 = lam0;\n        self.alpha_min = alpha_min;\n        self.e_alpha = e_alpha;\n        self.dc_alpha = dc_alpha;\n        self.e_ema = 0.0;\n        self.e_var = 0.05;\n        self.e_init = 0;\n        self.streak = 0;\n        self.dc_streak = 0;\n        self.sh_alarm_streak = 0;\n        self.en_alarm_streak = 0;\n        self.lv_alarm_streak = 0;\n        for j in 0..MAX_D {\n            self.dc_ema[j] = 0.0;\n            self.dc_var[j] = 0.05;\n            self.prev_dc[j] = 0.0;\n        }\n        let wm1 = w - 1;\n        let mut m = 0usize;\n        for row_k in 0..wm1 { m += if row_k < d { row_k + 1 } else { d }; }\n        self.m = m;\n        for i in 0..(w * wm1) { self.b_dct[i] = b_dct[i]; }\n        for k in 0..MAX_K { self.counts[k] = 0; }\n    }\n\n    #[inline]\n    pub fn canonicalize(&self, x_win: &[f32], u_out: &mut [f32; MAX_M], dc_out: &mut [f32; MAX_D]) -> f32 {\n        let w = self.w;\n        let d = self.d;\n        let wm1 = w - 1;\n        let mut xc = [0.0f32; MAX_W * MAX_D];\n        let mut norm_sq = 0.0f32;\n        for j in 0..d {\n            let mut mean = 0.0f32;\n            for t in 0..w { mean += x_win[t * d + j]; }\n            mean /= w as f32;\n            dc_out[j] = mean;\n            for t in 0..w {\n                let v = x_win[t * d + j] - mean;\n                xc[t * d + j] = v;\n                norm_sq += v * v;\n            }\n        }\n        let norm_raw = sqrt_f32(norm_sq);\n        let log_e = ln_f32(norm_raw + 1e-6);\n        if norm_raw < 1e-12 {\n            for i in 0..self.m { u_out[i] = 0.0; }\n            return log_e;\n        }\n        let inv_norm = 1.0 / norm_raw;\n\n        let mut r_mat = [0.0f32; MAX_D * MAX_W];\n        for j in 0..d {\n            for k in 0..wm1 {\n                let mut acc = 0.0f32;\n                for t in 0..w { acc += self.b_dct[t * wm1 + k] * (xc[t * d + j] * inv_norm); }\n                r_mat[j * wm1 + k] = acc;\n            }\n        }\n\n        for col in 0..d {\n            let mut row = d - 1;\n            while row > col {\n                let a = r_mat[(row - 1) * wm1 + col];\n                let b = r_mat[row * wm1 + col];\n                let rad = sqrt_f32(a * a + b * b);\n                if rad > 1e-7 {\n                    let c = a / rad;\n                    let s = -b / rad;\n                    for k in col..wm1 {\n                        let r1 = r_mat[(row - 1) * wm1 + k];\n                        let r2 = r_mat[row * wm1 + k];\n                        r_mat[(row - 1) * wm1 + k] = c * r1 - s * r2;\n                        r_mat[row * wm1 + k] = s * r1 + c * r2;\n                    }\n                    r_mat[row * wm1 + col] = 0.0;\n                }\n                row -= 1;\n            }\n        }\n\n        let mut det_sign: i32 = 1;\n        let mut col_sign = [1.0f32; MAX_D];\n        for i in 0..(d - 1) {\n            if r_mat[i * wm1 + i] < 0.0 {\n                col_sign[i] = -1.0;\n                det_sign = -det_sign;\n            }\n        }\n        if det_sign < 0 { col_sign[d - 1] = -1.0; }\n\n        let mut idx = 0usize;\n        for row_k in 0..wm1 {\n            let max_j = if row_k < d { row_k + 1 } else { d };\n            for col_j in 0..max_j {\n                u_out[idx] = r_mat[col_j * wm1 + row_k] * col_sign[col_j];\n                idx += 1;\n            }\n        }\n        log_e\n    }\n\n    #[inline]\n    fn dist_to(&self, u: &[f32; MAX_M], k: usize) -> f32 {\n        let m = self.m;\n        let mu = &self.means[k];\n        let p = &self.precs[k];\n        let mut diff = [0.0f32; MAX_M];\n        for i in 0..m { diff[i] = u[i] - mu[i]; }\n        let mut acc = 0.0f32;\n        for i in 0..m {\n            let mut dot = 0.0f32;\n            let row = i * m;\n            for j in 0..m { dot += p[row + j] * diff[j]; }\n            acc += diff[i] * dot;\n        }\n        if acc < 0.0 || !acc.is_finite() { 0.0 } else { acc }\n    }\n\n    #[inline]\n    fn spawn_proto(&mut self, u: &[f32; MAX_M]) {\n        let k = self.k_active;\n        let m = self.m;\n        let inv_lam = 1.0 / self.lam0;\n        for i in 0..m {\n            self.means[k][i] = u[i];\n            for j in 0..m { self.precs[k][i * m + j] = if i == j { inv_lam } else { 0.0 }; }\n        }\n        self.counts[k] = 1;\n        self.k_active += 1;\n    }\n\n    #[inline]\n    fn sm_update(&mut self, k: usize, u: &[f32; MAX_M]) {\n        let m = self.m;\n        self.counts[k] = self.counts[k].saturating_add(1);\n        let a_cnt = 1.0 / ((self.counts[k] as f32) + 4.0);\n        let a = if a_cnt > self.alpha_min { a_cnt } else { self.alpha_min };\n        let inv_1ma = 1.0 / (1.0 - a);\n\n        let mut diff = [0.0f32; MAX_M];\n        for i in 0..m { diff[i] = u[i] - self.means[k][i]; }\n\n        let mut pv = [0.0f32; MAX_M];\n        let mut q = 0.0f32;\n        for i in 0..m {\n            let mut dot = 0.0f32;\n            let row = i * m;\n            for j in 0..m { dot += self.precs[k][row + j] * diff[j]; }\n            pv[i] = dot;\n            q += diff[i] * dot;\n        }\n        let denom = (1.0 - a) + a * q;\n        if denom > 1e-6 && denom.is_finite() {\n            let scale = a / denom;\n            let max_p = 1.0e5f32;\n            for i in 0..m {\n                let row = i * m;\n                let pvi = pv[i] * scale;\n                for j in i..m {\n                    let mut val = (self.precs[k][row + j] - pvi * pv[j]) * inv_1ma;\n                    if val > max_p { val = max_p; } else if val < -max_p { val = -max_p; }\n                    self.precs[k][row + j] = val;\n                    self.precs[k][j * m + i] = val;\n                }\n            }\n        }\n        for i in 0..m { self.means[k][i] += a * diff[i]; }\n    }\n\n    /// Returns `(s_shape, s_energy, s_level, alarm_dual)`.\n    /// - If `huber_clip > 0.0` (Server Telemetry mode): uses Huber-clamped EMA on energy & DC level -> tracks slow normal drift WITHOUT swallowing sustained incidents!\n    /// - If `huber_clip <= 0.0` (Thermal Chip mode): uses fast ramp EMA + 3-win energy hysteresis + `dc_relock` stable-plateau re-lock.\n    #[inline]\n    pub fn step_v61(\n        &mut self, x_win: &[f32], learn: u8, shape_gate: f32, adapt_env: u8,\n        imp_thr: f32, lvl_thr: f32, sev_mult: f32, dc_relock: u8, huber_clip: f32,\n    ) -> (f32, f32, f32, u8) {\n        let mut u = [0.0f32; MAX_M];\n        let mut dc = [0.0f32; MAX_D];\n        let log_e = self.canonicalize(x_win, &mut u, &mut dc);\n\n        let (s_energy, s_level) = if self.e_init == 0 {\n            (0.0, 0.0)\n        } else {\n            let de = log_e - self.e_ema;\n            let se = (de * de) / (self.e_var + 0.05);\n            let mut sl = 0.0f32;\n            for j in 0..self.d {\n                let dd = dc[j] - self.dc_ema[j];\n                sl += (dd * dd) / (self.dc_var[j] + 0.05);\n            }\n            (se, sl / (self.d as f32))\n        };\n\n        let (best_k, s_shape) = if self.k_active == 0 {\n            (0usize, 0.0)\n        } else {\n            let mut bk = 0usize;\n            let mut bd = self.dist_to(&u, 0);\n            for k in 1..self.k_active {\n                let dk = self.dist_to(&u, k);\n                if dk < bd { bd = dk; bk = k; }\n            }\n            (bk, bd)\n        };\n\n        let shape_ok = self.k_active == 0 || s_shape <= shape_gate;\n\n        if learn != 0 && shape_ok {\n            if self.k_active == 0 {\n                self.spawn_proto(&u);\n            } else if self.k_active < self.k_max && s_shape > (0.35 / self.lam0) {\n                self.spawn_proto(&u);\n            } else {\n                self.sm_update(best_k, &u);\n            }\n        }\n\n        if (learn != 0 || adapt_env != 0) && shape_ok {\n            if self.e_init == 0 {\n                self.e_ema = log_e;\n                for j in 0..self.d {\n                    self.dc_ema[j] = dc[j];\n                    self.prev_dc[j] = dc[j];\n                }\n                self.e_init = 1;\n                self.streak = 0;\n                self.dc_streak = 0;\n            } else if huber_clip > 0.0 && learn == 0 {\n                // Robust Huber-Clamped EMA: tracks slow normal baseline drift without swallowing sustained anomalies\n                let ce = huber_clip * sqrt_f32(self.e_var + 0.05);\n                let de_c = clamp_f32(log_e - self.e_ema, -ce, ce);\n                self.e_ema += self.e_alpha * de_c;\n                self.e_var = (1.0 - self.e_alpha) * self.e_var + self.e_alpha * (de_c * de_c);\n\n                for j in 0..self.d {\n                    let cd = huber_clip * sqrt_f32(self.dc_var[j] + 0.05);\n                    let dd_c = clamp_f32(dc[j] - self.dc_ema[j], -cd, cd);\n                    self.dc_ema[j] += self.dc_alpha * dd_c;\n                    self.dc_var[j] = (1.0 - self.dc_alpha) * self.dc_var[j] + self.dc_alpha * (dd_c * dd_c);\n                }\n            } else {\n                // Fast thermal-ramp EMA + hysteresis re-lock\n                if s_energy <= imp_thr || learn != 0 {\n                    self.streak = 0;\n                    let de = log_e - self.e_ema;\n                    self.e_ema += self.e_alpha * de;\n                    self.e_var = (1.0 - self.e_alpha) * self.e_var + self.e_alpha * (de * de);\n                } else {\n                    self.streak = self.streak.saturating_add(1);\n                    if self.streak >= 3 {\n                        self.e_ema = log_e;\n                        self.streak = 0;\n                    }\n                }\n                if s_level <= lvl_thr || learn != 0 {\n                    self.dc_streak = 0;\n                    for j in 0..self.d {\n                        let dd = dc[j] - self.dc_ema[j];\n                        self.dc_ema[j] += self.dc_alpha * dd;\n                        self.dc_var[j] = (1.0 - self.dc_alpha) * self.dc_var[j] + self.dc_alpha * (dd * dd);\n                    }\n                } else if dc_relock > 0 {\n                    let mut step_sq = 0.0f32;\n                    for j in 0..self.d {\n                        let dprev = dc[j] - self.prev_dc[j];\n                        step_sq += dprev * dprev;\n                    }\n                    step_sq /= self.d as f32;\n                    if step_sq < 1.0 {\n                        self.dc_streak = self.dc_streak.saturating_add(1);\n                        if self.dc_streak >= dc_relock {\n                            for j in 0..self.d { self.dc_ema[j] = dc[j]; }\n                            self.dc_streak = 0;\n                        }\n                    } else {\n                        self.dc_streak = 0;\n                    }\n                }\n            }\n            for j in 0..self.d { self.prev_dc[j] = dc[j]; }\n        }\n\n        let mut alarm_dual: u8 = 0;\n        if learn == 0 && shape_gate < 1e10 {\n            self.sh_alarm_streak = if s_shape > shape_gate { self.sh_alarm_streak.saturating_add(1) } else { 0 };\n            self.en_alarm_streak = if s_energy > imp_thr { self.en_alarm_streak.saturating_add(1) } else { 0 };\n            self.lv_alarm_streak = if s_level > lvl_thr { self.lv_alarm_streak.saturating_add(1) } else { 0 };\n\n            let trip_sh = (s_shape > shape_gate * sev_mult) || (self.sh_alarm_streak >= 2);\n            let trip_en = (s_energy > imp_thr * sev_mult) || (self.en_alarm_streak >= 2);\n            let trip_lv = (s_level > lvl_thr * sev_mult) || (self.lv_alarm_streak >= 2);\n            if trip_sh || trip_en || trip_lv { alarm_dual = 1; }\n        }\n\n        (s_shape, s_energy, s_level, alarm_dual)\n    }\n}\n\n#[no_mangle]\npub extern "C" fn ppc_triad_engine_bytes() -> usize {\n    core::mem::size_of::<GaugeTriadEngineF32>()\n}\n\n#[no_mangle]\npub unsafe extern "C" fn ppc_triad_init(\n    eng: *mut GaugeTriadEngineF32, w: usize, d: usize, k_max: usize,\n    lam0: f32, alpha_min: f32, e_alpha: f32, dc_alpha: f32, b_dct: *const f32,\n) {\n    let b_s = core::slice::from_raw_parts(b_dct, w * (w - 1));\n    (*eng).init(w, d, k_max, lam0, alpha_min, e_alpha, dc_alpha, b_s);\n}\n\n#[no_mangle]\npub unsafe extern "C" fn ppc_triad_run_stream_v61(\n    eng: *mut GaugeTriadEngineF32, x_batch: *const f32, n_samples: usize,\n    learn: u8, shape_gate: f32, adapt_env: u8, imp_thr: f32, lvl_thr: f32,\n    sev_mult: f32, dc_relock: u8, huber_clip: f32,\n    out_shape: *mut f32, out_energy: *mut f32, out_level: *mut f32, out_alarm: *mut u8,\n) {\n    let e = &mut *eng;\n    let stride = e.w * e.d;\n    let x_all = core::slice::from_raw_parts(x_batch, n_samples * stride);\n    let sh_s = core::slice::from_raw_parts_mut(out_shape, n_samples);\n    let en_s = core::slice::from_raw_parts_mut(out_energy, n_samples);\n    let lv_s = core::slice::from_raw_parts_mut(out_level, n_samples);\n    let al_s = core::slice::from_raw_parts_mut(out_alarm, n_samples);\n    for i in 0..n_samples {\n        let win = &x_all[i * stride..(i + 1) * stride];\n        let (sh, en, lv, al) = e.step_v61(win, learn, shape_gate, adapt_env, imp_thr, lvl_thr, sev_mult, dc_relock, huber_clip);\n        sh_s[i] = sh; en_s[i] = en; lv_s[i] = lv; al_s[i] = al;\n    }\n}\n')

subprocess.run("cd ppc_gauge_triad_v061 && cargo build --release -q", shell=True, check=True)

lib = ctypes.CDLL("ppc_gauge_triad_v061/target/release/libppc_gauge_triad_rs.so")
lib.ppc_triad_engine_bytes.restype = ctypes.c_size_t
lib.ppc_triad_init.argtypes = [
    ctypes.c_void_p, ctypes.c_size_t, ctypes.c_size_t, ctypes.c_size_t,
    ctypes.c_float, ctypes.c_float, ctypes.c_float, ctypes.c_float, ctypes.POINTER(ctypes.c_float)
]
lib.ppc_triad_run_stream_v61.argtypes = [
    ctypes.c_void_p, ctypes.POINTER(ctypes.c_float), ctypes.c_size_t,
    ctypes.c_uint8, ctypes.c_float, ctypes.c_uint8, ctypes.c_float, ctypes.c_float,
    ctypes.c_float, ctypes.c_uint8, ctypes.c_float,
    ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float), ctypes.POINTER(ctypes.c_uint8)
]

def compute_auroc(y_true, scores):
    y_true = np.asarray(y_true, dtype=np.int32)
    scores = np.asarray(scores, dtype=np.float64)
    pos, neg = scores[y_true == 1], scores[y_true == 0]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    if len(pos) * len(neg) > 4_000_000:
        rng = np.random.default_rng(0)
        pos = rng.choice(pos, min(len(pos), 2000), replace=False)
        neg = rng.choice(neg, min(len(neg), 2000), replace=False)
    return float(np.mean((pos[:, None] - neg[None, :] > 0) + 0.5 * (pos[:, None] - neg[None, :] == 0)))

def conformal_threshold(scores_cal, alpha=0.05):
    s = np.sort(np.asarray(scores_cal, dtype=np.float64))
    return float(s[max(0, min(len(s) - 1, int(np.ceil((1.0 - alpha) * (len(s) + 1))) - 1))])

def make_dct_f32(w):
    n = np.arange(w)[:, None]; k = np.arange(1, w)[None, :]
    B = np.cos(np.pi * (n + 0.5) * k / w)
    return np.ascontiguousarray(B / np.linalg.norm(B, axis=0, keepdims=True), dtype=np.float32)

class RustTriadV061:
    def __init__(self, w=8, d=4, k_max=3, lam0=5e-3, alpha_min=0.01, e_alpha=0.008, dc_alpha=0.004):
        self.sz = int(lib.ppc_triad_engine_bytes())
        self.buf = ctypes.create_string_buffer(self.sz)
        self.B = make_dct_f32(w)
        lib.ppc_triad_init(
            ctypes.byref(self.buf), w, d, k_max,
            ctypes.c_float(lam0), ctypes.c_float(alpha_min),
            ctypes.c_float(e_alpha), ctypes.c_float(dc_alpha),
            self.B.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
        )
    def run(self, X_wins, learn=False, shape_gate=1e20, adapt_env=False, imp_thr=10.0, lvl_thr=10.0, sev_mult=2.5, dc_relock=0, huber_clip=1.5):
        Xc = np.ascontiguousarray(X_wins, dtype=np.float32); n = len(Xc)
        sh = np.empty(n, np.float32); en = np.empty(n, np.float32); lv = np.empty(n, np.float32)
        al = np.empty(n, np.uint8)
        lib.ppc_triad_run_stream_v61(
            ctypes.byref(self.buf), Xc.ctypes.data_as(ctypes.POINTER(ctypes.c_float)), n,
            1 if learn else 0, ctypes.c_float(shape_gate), 1 if adapt_env else 0,
            ctypes.c_float(imp_thr), ctypes.c_float(lvl_thr),
            ctypes.c_float(sev_mult), int(dc_relock), ctypes.c_float(huber_clip),
            sh.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            en.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            lv.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            al.ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))
        )
        return sh.astype(np.float64), en.astype(np.float64), lv.astype(np.float64), al.astype(bool)

def load_smd_raw(mach):
    base = "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset"
    return (np.loadtxt(f"{base}/train/{mach}.txt", delimiter=","),
            np.loadtxt(f"{base}/test/{mach}.txt", delimiter=","),
            np.loadtxt(f"{base}/test_label/{mach}.txt", delimiter=","))

def load_psm_raw():
    base = "https://raw.githubusercontent.com/eBay/RANSynCoders/main/data"
    return (pd.read_csv(f"{base}/train.csv").iloc[:,1:].fillna(0).to_numpy(float),
            pd.read_csv(f"{base}/test.csv").iloc[:,1:].fillna(0).to_numpy(float),
            pd.read_csv(f"{base}/test_label.csv").iloc[:,1].to_numpy(int))

def to_windows_v6(tr_raw, te_raw, yl_raw, w=8, d=4, max_tr_win=2000, use_pca_resid=True):
    mu, sd = tr_raw.mean(0), np.maximum(tr_raw.std(0), 1e-6)
    Ztr_std, Zte_std = (tr_raw - mu) / sd, (te_raw - mu) / sd
    if use_pca_resid and tr_raw.shape[1] > d:
        pca = PCA(n_components=d - 1, random_state=0).fit(Ztr_std)
        Ptr, Pte = pca.transform(Ztr_std), pca.transform(Zte_std)
        rtr = np.linalg.norm(Ztr_std - pca.inverse_transform(Ptr), axis=1, keepdims=True)
        rte = np.linalg.norm(Zte_std - pca.inverse_transform(Pte), axis=1, keepdims=True)
        r_mu, r_sd = rtr.mean(0, keepdims=True), np.maximum(rtr.std(0, keepdims=True), 1e-6)
        Ztr = np.hstack([Ptr, (rtr - r_mu) / r_sd])
        Zte = np.hstack([Pte, (rte - r_mu) / r_sd])
        meta = (mu, sd, pca, r_mu, r_sd)
    else:
        pca = PCA(n_components=d, random_state=0).fit(Ztr_std)
        Ztr, Zte = pca.transform(Ztr_std), pca.transform(Zte_std)
        meta = (mu, sd, pca, None, None)
    ntr, nte = min(max_tr_win, len(Ztr) // w), len(Zte) // w
    return Ztr[:ntr*w].reshape(ntr, w, d), Zte[:nte*w].reshape(nte, w, d), (yl_raw[:nte*w].reshape(nte, w).max(1) > 0).astype(np.int32), meta

def triad_norm_score(sh_c, en_c, lv_c, sh_t, en_t, lv_t):
    def rz(c, t):
        lc, lt = np.log1p(c), np.log1p(t)
        med = np.median(lc); mad = np.median(np.abs(lc - med)) * 1.4826 + 0.1
        return np.maximum(0.0, (lt - med) / mad)
    return rz(sh_c, sh_t) + rz(en_c, en_t) + rz(lv_c, lv_t)

def dual_gate_scalar(scores, thr, sev_mult=2.5):
    al = np.zeros(len(scores), dtype=bool); st = 0
    for i, s in enumerate(scores):
        st = st + 1 if s > thr else 0
        if (s > thr * sev_mult) or (st >= 2): al[i] = True
    return al

print(f"Rust PPC-Gauge-Triad v0.6.1 compiled! Struct SRAM = {lib.ppc_triad_engine_bytes()} bytes ({lib.ppc_triad_engine_bytes()/1024:.2f} KB)")


## Cell 2 — Ablasi Bersih 8 Server Nyata: v0.5 vs v0.6.1-A (PCA-4 + Huber-EMA) vs v0.6.1-B (3PC+Resid + Huber-EMA)


In [ ]:
ALL_SERVERS = [
    ("SMD machine-1-1", lambda: load_smd_raw("machine-1-1")),
    ("SMD machine-1-6", lambda: load_smd_raw("machine-1-6")),
    ("SMD machine-2-1", lambda: load_smd_raw("machine-2-1")),
    ("PSM eBay Server", load_psm_raw),
    ("SMD machine-1-2", lambda: load_smd_raw("machine-1-2")),
    ("SMD machine-1-3", lambda: load_smd_raw("machine-1-3")),
    ("SMD machine-2-2", lambda: load_smd_raw("machine-2-2")),
    ("SMD machine-3-1", lambda: load_smd_raw("machine-3-1")),
]

def eval_cfg(tr_r, te_r, yl_r, use_resid, adapt_env, huber_clip, use_dual, e_a, dc_a):
    Wtr, Wte, yte, _ = to_windows_v6(tr_r, te_r, yl_r, use_pca_resid=use_resid)
    kf = int(len(Wtr) * 0.75)
    eng = RustTriadV061(8, 4, k_max=3, lam0=5e-3, alpha_min=0.01, e_alpha=e_a, dc_alpha=dc_a)
    eng.run(Wtr[:kf], learn=True)
    shc, enc, lvc, _ = eng.run(Wtr[kf:], learn=False)
    thr_sh = conformal_threshold(shc, 0.05)
    thr_sc = conformal_threshold(triad_norm_score(shc, enc, lvc, shc, enc, lvc), 0.05)
    sht, ent, lvt, _ = eng.run(Wte, learn=False, shape_gate=thr_sh, adapt_env=adapt_env, dc_relock=0, huber_clip=huber_clip)
    sc = triad_norm_score(shc, enc, lvc, sht, ent, lvt)
    al = dual_gate_scalar(sc, thr_sc, 2.5) if use_dual else (sc > thr_sc)
    return compute_auroc(yte, sc)*100, np.mean(al[yte==0])*100, np.mean(al[yte==1])*100

print(f"{'Server':<16} | {'v0.5 (NoAdapt)':^20} | {'v0.6.1A (PCA4+Huber)':^20} | {'v0.6.1B (3PC+R+Huber)':^21}")
print(f"{'':<16} | {'AUROC':>6} {'FAR':>6} {'Rec':>6} | {'AUROC':>6} {'FAR':>6} {'Rec':>6} | {'AUROC':>6} {'FAR':>6} {'Rec':>6}")
print("-" * 88)

res = {k: [[], [], []] for k in ("v5", "v61a", "v61b")}
cache_raw = {}
for name, loader in ALL_SERVERS:
    tr_r, te_r, yl_r = loader()
    cache_raw[name] = (tr_r, te_r, yl_r)
    r5 = eval_cfg(tr_r, te_r, yl_r, False, False, 0.0, False, 0.10, 0.05)
    ra = eval_cfg(tr_r, te_r, yl_r, False, True,  1.5, True,  0.008, 0.004)
    rb = eval_cfg(tr_r, te_r, yl_r, True,  True,  1.5, True,  0.008, 0.004)
    for k, rv in (("v5", r5), ("v61a", ra), ("v61b", rb)):
        for i in range(3): res[k][i].append(rv[i])
    print(f"{name:<16} | {r5[0]:5.1f}% {r5[1]:5.1f}% {r5[2]:5.1f}% | {ra[0]:5.1f}% {ra[1]:5.1f}% {ra[2]:5.1f}% | {rb[0]:5.1f}% {rb[1]:5.1f}% {rb[2]:5.1f}%")

print("-" * 88)
m5, ma, mb = [[np.mean(res[k][i]) for i in range(3)] for k in ("v5", "v61a", "v61b")]
print(f"{'8-SERVER MEAN':<16} | {m5[0]:5.1f}% {m5[1]:5.1f}% {m5[2]:5.1f}% | {ma[0]:5.1f}% {ma[1]:5.1f}% {ma[2]:5.1f}% | {mb[0]:5.1f}% {mb[1]:5.1f}% {mb[2]:5.1f}%")


## Cell 3 — Test 3 (v0.6.1 Huber-EMA): Zero-Shot vs Zero-Touch On-Device Auto-Warmup (100/200/400w)


In [ ]:
print("=== TEST 3 (v0.6.1 Huber-EMA): CROSS-DEVICE AUTO-WARMUP CURVE ===")
tr11, te11, yl11 = cache_raw["SMD machine-1-1"]
Wtr_src, _, _, (mu_s, sd_s, pca_s, rmu_s, rsd_s) = to_windows_v6(tr11, te11, yl11, use_pca_resid=True)
kf_s = int(len(Wtr_src) * 0.75)
e_src = RustTriadV061(8, 4, k_max=3, lam0=5e-3, alpha_min=0.01, e_alpha=0.008, dc_alpha=0.004)
e_src.run(Wtr_src[:kf_s], learn=True)
sh_cs, en_cs, lv_cs, _ = e_src.run(Wtr_src[kf_s:], learn=False)

WARM_COUNTS = [0, 100, 200, 400]
print(f"{'Unseen Target':<18}" + "".join(f"{('ZeroShot' if n==0 else f'AutoWarm {n}w'):>14}" for n in WARM_COUNTS))
print("-" * (18 + 14 * len(WARM_COUNTS)))

cols = {n: [] for n in WARM_COUNTS}
for tgt in ["SMD machine-1-2", "SMD machine-1-6", "SMD machine-2-1"]:
    tr_t, te_t, yl_t = cache_raw[tgt]
    row = f"{tgt:<18}"
    for nw in WARM_COUNTS:
        if nw == 0:
            Z_std = (te_t - mu_s) / sd_s
            P_t = pca_s.transform(Z_std)
            r_t = np.linalg.norm(Z_std - pca_s.inverse_transform(P_t), axis=1, keepdims=True)
            Z_t = np.hstack([P_t, (r_t - rmu_s) / rsd_s])
            nte = len(Z_t) // 8
            Wt = Z_t[:nte*8].reshape(nte, 8, 4)
            yt = (yl_t[:nte*8].reshape(nte, 8).max(1) > 0).astype(np.int32)
            ee = RustTriadV061(8, 4, k_max=3, lam0=5e-3, alpha_min=0.01, e_alpha=0.008, dc_alpha=0.004)
            ee.run(Wtr_src[:kf_s], learn=True)
            sht, ent, lvt, _ = ee.run(Wt, learn=False, adapt_env=True, huber_clip=1.5)
            au = compute_auroc(yt, triad_norm_score(sh_cs, en_cs, lv_cs, sht, ent, lvt)) * 100
        else:
            Wtr_t, Wte_t, yte_t, _ = to_windows_v6(tr_t[:nw*8], te_t, yl_t, max_tr_win=nw, use_pca_resid=True)
            kf = int(len(Wtr_t) * 0.75)
            ee = RustTriadV061(8, 4, k_max=3, lam0=5e-3, alpha_min=0.01, e_alpha=0.008, dc_alpha=0.004)
            ee.run(Wtr_t[:kf], learn=True)
            shc, enc, lvc, _ = ee.run(Wtr_t[kf:], learn=False)
            thr_sh = conformal_threshold(shc, 0.05)
            sht, ent, lvt, _ = ee.run(Wte_t, learn=False, shape_gate=thr_sh, adapt_env=True, huber_clip=1.5)
            au = compute_auroc(yte_t, triad_norm_score(shc, enc, lvc, sht, ent, lvt)) * 100
        cols[nw].append(au)
        row += f"{au:13.1f}%"
    print(row)
print("-" * (18 + 14 * len(WARM_COUNTS)))
print(f"{'MEAN AUROC':<18}" + "".join(f"{np.mean(cols[n]):13.1f}%" for n in WARM_COUNTS))
